# 02 - Data Preprocessing & Cleaning

> **Project:** AI-Based Interview Preparation Assistant  
> **Course:** AI252TA – Machine Learning Operations (Semester V)  
> **Status:** Phase 1 – Completed  

### Objective & Purpose
Demonstrate text cleaning, grouped splitting, and reproducibility of the preprocessing pipeline.

---


In [1]:
import json
import re
import yaml
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'params.yaml').exists())
params = yaml.safe_load((PROJECT_ROOT / 'params.yaml').read_text())

print(f'Project root: {PROJECT_ROOT}')

Project root: /home/likithshashidhar/MLOps


## Before/After Text Cleaning Examples

In [2]:
# Load raw data (before cleaning)
raw_path = PROJECT_ROOT / params['data']['raw_data_path']
raw_df = pd.read_parquet(raw_path)

# Load processed train data (after cleaning)
train_path = PROJECT_ROOT / 'data/processed/train.parquet'
train_df = pd.read_parquet(train_path)

print(f'Raw rows: {len(raw_df)}')
print(f'Train rows: {len(train_df)}')
print(f'Train unique questions: {train_df["question_id"].nunique()}')

Raw rows: 2273
Train rows: 1561
Train unique questions: 56


In [3]:
# Show before/after examples
sample_idx = 0
raw_row = raw_df.iloc[sample_idx]

# Find corresponding row in train (by question_id and candidate_answer prefix)
qid = raw_row['question_id']
cand_prefix = raw_row['candidate_answer'][:50]
train_row = train_df[(train_df['question_id'] == qid) & 
                     (train_df['candidate_answer'].str.startswith(cand_prefix))]

if len(train_row) > 0:
    train_row = train_row.iloc[0]
    print('=== BEFORE (raw) ===')
    print(f'question: {raw_row["question"][:200]}...')
    print(f'reference_answer: {raw_row["reference_answer"][:200]}...')
    print(f'candidate_answer: {raw_row["candidate_answer"][:200]}...')
    print()
    print('=== AFTER (cleaned) ===')
    print(f'question: {train_row["question"][:200]}...')
    print(f'reference_answer: {train_row["reference_answer"][:200]}...')
    print(f'candidate_answer: {train_row["candidate_answer"][:200]}...')

In [4]:
# Show what cleaning does: collapse whitespace, strip, no lowercasing
def clean_text(text):
    if pd.isna(text):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

examples = [
    "  Hello    world  ",
    "Line 1\n\nLine 2\t\tTabbed",
    "Multiple   spaces    and\nnewlines",
    "Already clean text.",
]

for ex in examples:
    print(f'Before: {repr(ex)}')
    print(f'After:  {repr(clean_text(ex))}')
    print()

Before: '  Hello    world  '
After:  'Hello world'

Before: 'Line 1\n\nLine 2\t\tTabbed'
After:  'Line 1 Line 2 Tabbed'

Before: 'Multiple   spaces    and\nnewlines'
After:  'Multiple spaces and newlines'

Before: 'Already clean text.'
After:  'Already clean text.'



> **Takeaway:** Text cleaning strips leading/trailing whitespace, collapses multiple spaces/newlines/tabs into single spaces, and preserves case. No lowercasing is applied per the preprocessing specification.

## Grouped Split Verification

In [5]:
# Load all splits
val_df = pd.read_parquet(PROJECT_ROOT / 'data/processed/val.parquet')
test_df = pd.read_parquet(PROJECT_ROOT / 'data/processed/test.parquet')

train_qids = set(train_df['question_id'].unique())
val_qids = set(val_df['question_id'].unique())
test_qids = set(test_df['question_id'].unique())

print(f'Train questions: {len(train_qids)}')
print(f'Val questions: {len(val_qids)}')
print(f'Test questions: {len(test_qids)}')
print(f'Total unique: {len(train_qids | val_qids | test_qids)}')

# Assert zero overlap
assert train_qids.isdisjoint(val_qids), 'Train/Val overlap!'
assert train_qids.isdisjoint(test_qids), 'Train/Test overlap!'
assert val_qids.isdisjoint(test_qids), 'Val/Test overlap!'
print('✓ Zero question_id overlap across splits')

Train questions: 56
Val questions: 8
Test questions: 17
Total unique: 81
✓ Zero question_id overlap across splits


In [6]:
# Verify row count preservation
total_raw = len(raw_df)
total_split = len(train_df) + len(val_df) + len(test_df)
print(f'Raw rows: {total_raw}')
print(f'Split rows: {total_split}')
assert total_raw == total_split, f'Row count mismatch: {total_raw} != {total_split}'
print('✓ All rows preserved in splits')

Raw rows: 2273
Split rows: 2273
✓ All rows preserved in splits


## Per-Split Class Distribution

In [7]:
target_classes = params['data']['target_classes']

splits = {
    'train': train_df,
    'val': val_df,
    'test': test_df
}

for name, df in splits.items():
    dist = df['quality_label'].value_counts().sort_index()
    pct = (dist / len(df) * 100).round(2)
    print(f'\n{name.upper()} ({len(df)} rows, {df["question_id"].nunique()} questions):')
    for cls, count in dist.items():
        print(f'  Class {cls} ({target_classes[cls]}): {count} ({pct[cls]:.1f}%)')


TRAIN (1561 rows, 56 questions):
  Class 0 (Poor): 62 (4.0%)
  Class 1 (Needs Improvement): 233 (14.9%)
  Class 2 (Good): 277 (17.8%)
  Class 3 (Excellent): 989 (63.4%)

VAL (229 rows, 8 questions):
  Class 0 (Poor): 8 (3.5%)
  Class 1 (Needs Improvement): 38 (16.6%)
  Class 2 (Good): 33 (14.4%)
  Class 3 (Excellent): 150 (65.5%)

TEST (483 rows, 17 questions):
  Class 0 (Poor): 10 (2.1%)
  Class 1 (Needs Improvement): 99 (20.5%)
  Class 2 (Good): 86 (17.8%)
  Class 3 (Excellent): 288 (59.6%)


In [8]:
# Plot class distribution across splits
fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, (name, df) in zip(axes, splits.items()):
    dist = df['quality_label'].value_counts().sort_index()
    pct = dist / len(df) * 100
    ax.bar([target_classes[i] for i in dist.index], pct.values, color=['#4c72b0', '#55a868', '#c44e52', '#8172b3'])
    ax.set_title(f'{name.capitalize()}')
    ax.set_ylabel('Percentage')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

/tmp/ipykernel_67821/1454158933.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


> **Takeaway:** Grouped splitting by question_id ensures zero question leakage. Class distributions are roughly preserved across splits despite the grouping constraint (which can cause some variance since entire questions go to one split). The 'Excellent' class dominates in all splits.

## Reproducibility Check

In [9]:
# Run the real grouped split function from src/ to verify reproducibility
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from data.preprocess import grouped_split, clean_text_columns

# Run split with same params
split_params = params['split']
train2, val2, test2 = grouped_split(
    df=raw_df,
    group_column=split_params['group_column'],
    test_size=split_params['test_size'],
    validation_size=split_params['validation_size'],
    random_state=split_params['random_state'],
)

# Compare question_id sets
print('Original train qids == New train qids:', 
      set(train_df['question_id']) == set(train2['question_id']))
print('Original val qids == New val qids:', 
      set(val_df['question_id']) == set(val2['question_id']))
print('Original test qids == New test qids:', 
      set(test_df['question_id']) == set(test2['question_id']))

print('\n✓ Split is reproducible with same random_state')

Original train qids == New train qids: True
Original val qids == New val qids: True
Original test qids == New test qids: True

✓ Split is reproducible with same random_state


> **Takeaway:** The grouped split is fully reproducible with the same random_state (42). Using the actual `grouped_split` function from `src/data/preprocess.py` produces identical splits, confirming pipeline reproducibility.